# Theory: convergence, consistency, and oracle properties

The theory of model-based boosting describes optimization convergence,
estimation as the sample grows, and adaptation to unknown sparsity:

1. **Convergence:** whether the boosting risk decreases at every step, and
   whether the path converges to a limit as $m_{\text{stop}} \to \infty$.
2. **Consistency:** whether the boosting estimator, stopped at the right
   iteration, converges to the true data-generating function as $n \to \infty$.
3. **Oracle properties:** whether boosting adapts to unknown sparsity,
   achieving good rates when only a few predictors matter, without knowing
   *which* ones.

The main results are stated with intuition and proof sketches; the cited
sources contain the full proofs and their conditions. The prerequisites are in
[](prerequisites.ipynb) and [](functional_gradient_boosting.ipynb).

```{admonition} Notation recap
:class: tip
We observe $\{(y_i, x_i)\}_{i=1}^n$. The empirical risk is
$\mathcal{R}_n(f) = \frac{1}{n}\sum_i \rho(y_i, f(x_i))$. The population risk is
$\mathcal{R}(f) = \mathbb{E}[\rho(Y, f(X))]$. The Bayes-optimal predictor is
$f^* = \argmin_f \mathcal{R}(f)$. The boosting estimator stopped at $m$ is
$\hat f^{[m]}$.
```



## 1. Convergence of the boosting path



### 1.1 Monotone decrease of the training risk

For a convex differentiable loss and a sufficiently small step length, each
boosting update does not increase the empirical risk.

```{admonition} Theorem 1 (monotone risk decrease)
:class: important

Let $\rho$ be convex and differentiable in $f$. Let $\hat g_{j^\star}$ be the
best-fitting base learner to the negative gradient $u^{[m]} = -\nabla \mathcal{R}_n(\hat f^{[m-1]})$.
For step length $\nu \in (0,1]$ sufficiently small,

$$
\mathcal{R}_n(\hat f^{[m]}) \le \mathcal{R}_n(\hat f^{[m-1]}).
$$

If $\rho$ is strictly convex and $\hat g_{j^\star} \ne 0$, the inequality is
strict.
```

**Proof sketch.** The negative gradient $u^{[m]}$ is a descent direction of
$\mathcal{R}_n$ at $\hat f^{[m-1]}$ (by definition of gradient). Fitting a base
learner $\hat g_{j^\star}$ to $u^{[m]}$ by least squares produces the
projection of $u^{[m]}$ onto the span of the learner, which is the *closest
realizable* descent direction. For small enough $\nu$, the step
$\hat f^{[m]} = \hat f^{[m-1]} + \nu \hat g_{j^\star}$ remains in the region
where the linear approximation is valid, so $\mathcal{R}_n$ decreases. A formal
Taylor-expansion argument bounds the decrease:

$$
\mathcal{R}_n(\hat f^{[m]}) \le \mathcal{R}_n(\hat f^{[m-1]}) - c\,\nu\, \|u^{[m]}\|^2 + O(\nu^2),
$$

where $c > 0$ depends on the angle between $\hat g_{j^\star}$ and $u^{[m]}$.
For $\nu$ small, the linear term dominates. $\square$

```{figure} ../figures/fig-risk-decrease.svg
:label: fig-risk-decrease
:alt: Monotone empirical risk decrease per boosting iteration. Risk drops quickly at first, then asymptotes to the best achievable risk as m_stop grows.

Empirical risk against boosting iteration. The drop is steep early, when the negative gradient is large, and flattens as the fit approaches the best the learner set can achieve.
```



### 1.2 The L2Boosting convergence result

For L2Boosting (Gaussian loss, $\rho(y,f) = \tfrac12(y-f)^2$), the convergence
is exact and quantifiable. Let $H_j$ be the hat matrix of learner $j$, and let
$j^\star_m$ be the learner selected at iteration $m$. The boosting hat matrix
$B_m$ satisfies the recursion (derived in [](functional_gradient_boosting.ipynb)):

$$
B_m = B_{m-1} + \nu\, H_{j^\star_m}\,(I - B_{m-1}).
$$

Writing the residual $r_m = y - B_m y = (I - B_m) y$, we get

$$
r_m = (I - \nu H_{j^\star_m})\, r_{m-1}.
$$

For the penalized least-squares learners used here, $0 \preceq H_j \preceq I$
holds (eigenvalues of $H_j$ lie in $[0,1]$ even though $H_j$ is not idempotent
once $\lambda > 0$), so the matrix $(I - \nu H_j)$ has eigenvalues in
$[1-\nu, 1]$. Therefore

$$
\|r_m\| \le \|r_{m-1}\|,
$$

with strict decrease whenever the selected learner captures any of the residual
($H_{j^\star_m} r_{m-1} \ne 0$). Over $m$ iterations:

$$
\|r_m\|^2 \le \|r_0\|^2 \prod_{\ell=1}^{m} \big(1 - \nu\, \alpha_\ell\big),
$$

where $\alpha_\ell = \|H_{j^\star_\ell} r_{\ell-1}\|^2 / \|r_{\ell-1}\|^2 \in [0,1]$
is the fraction of residual variance captured at step $\ell$.

```{admonition} Theorem 2 (L2Boosting convergence)
:class: important

This result is due to {cite}`buhlmann2003boosting`.

If the union of the base learners' column spaces spans $\mathbb{R}^n$ (or at
least contains $y$), then L2Boosting converges:

$$
\|\hat f^{[m]} - y\| \to 0 \qquad \text{as } m \to \infty,
$$

i.e., the boosted predictor interpolates the training data in the limit. The
convergence rate depends on how efficiently the greedy selection covers the
residual subspace.
```

Unstopped L2Boosting eventually fits the training data perfectly —
overfitting. Early stopping supplies the regularization.



### 1.3 The shrinkage factor and convergence speed

The step length $\nu$ controls how quickly the path converges (and thus how
fine-grained the regularization grid is). For L2Boosting with a single
repeatedly-selected learner $H$:

$$
r_m = (I - \nu H)^m\, r_0.
$$

The eigenvalues of $(I - \nu H)^m$ are $(1 - \nu h_\ell)^m$ where
$h_\ell \in [0,1]$ are the eigenvalues of $H$. A component with $h_\ell$ close to 1 (well-explained by the learner) decays
to zero in about $1/\nu$ steps; a component with $h_\ell$ close to 0 (barely
explained) persists for about $1/(\nu h_\ell)$ steps.

```{figure} ../figures/fig-convergence-nu.svg
:label: fig-convergence-nu
:alt: Residual energy versus iteration for three step lengths. Smaller nu converges more slowly but yields a finer grid of models; nu=0.1 is the default.

Residual energy against iteration for three step lengths. A smaller $\nu$
takes more iterations to reach the same fit and provides a finer grid of
stopping choices. The default is $\nu = 0.1$.
```

```{admonition} The ν-m_stop duality
:class: important

For L2Boosting, halving $\nu$ roughly **doubles** the optimal $m_{\text{stop}}$.
The two parameters are not independent: they trade off along a ridge of
near-equivalent test performance. The standard practice is to fix $\nu$ small
(0.1) and tune only $m_{\text{stop}}$, because:

1. $\nu = 0.1$ gives a fine enough grid for the optimal model to be well-resolved.
2. Tuning one parameter (by CV) is cheaper than tuning two.
3. Smaller $\nu$ gives slightly better test performance {cite}`buhlmann2007boosting`,
   but with diminishing returns below 0.1.
```



### 1.4 Numerical check: Theorem 1 and overfitting

This simulation checks two claims: training risk decreases under Theorem 1's
conditions, while test risk can rise after reaching a minimum. That increase
motivates early stopping. The response is $y = 2x_0 - x_1 + \varepsilon$ with
$\varepsilon \sim N(0,1)$ and 60 further pure-noise predictors, all standard
normal. One training sample of $n = 100$ is fitted by Gaussian L2Boosting with
a linear learner (`bols`) per predictor, step length $\nu = 0.1$ and
$m_{\max} = 2000$. Test risk is the mean squared error on 3000 independent
observations, evaluated every 20 iterations.

In [1]:
import warnings
import numpy as np
import mboostrs as mb

warnings.filterwarnings("ignore")

rng = np.random.default_rng(0)
n_train, n_test = 100, 3000
p_noise = 60


def gen(n, rng):
    X = rng.normal(size=(n, 2 + p_noise))
    signal = 2.0 * X[:, 0] - 1.0 * X[:, 1]
    return X, signal + rng.normal(scale=1.0, size=n)


X_train, y_train = gen(n_train, rng)
X_test, y_test = gen(n_test, rng)
cols = [f"x{i}" for i in range(X_train.shape[1])]
data_train = {c: X_train[:, i] for i, c in enumerate(cols)}
data_train["y"] = y_train
data_test = {c: X_test[:, i] for i, c in enumerate(cols)}

formula = "y ~ " + " + ".join(f"bols({c})" for c in cols)
mstop_max = 2000
fit = mb.glmboost(formula, data=data_train, family=mb.Gaussian(),
                   control=mb.boost_control(mstop=mstop_max, nu=0.1))

# Theorem 1: the training risk path is non-increasing.
train_risk = fit.risk_
print(f"training risk monotone non-increasing: {np.all(np.diff(train_risk) <= 1e-9)}")
assert np.all(np.diff(train_risk) <= 1e-9)

training risk monotone non-increasing: True


In [2]:
# The test-risk curve on the large held-out set.
grid = np.arange(0, mstop_max + 1, 20)
test_risk = np.array([
    np.mean((np.full(n_test, fit.offset_) - y_test) ** 2) if m == 0
    else np.mean((fit[:m].predict(newdata=data_test, type="response") - y_test) ** 2)
    for m in grid
])
m_opt = int(grid[np.argmin(test_risk)])
print(f"test risk: m=0 -> {test_risk[0]:.3f}, "
      f"min={test_risk.min():.3f} at m={m_opt}, "
      f"m={mstop_max} -> {test_risk[-1]:.3f}")

# The risk falls from the offset, reaches an interior minimum and rises again
# well before mstop_max.
assert test_risk[0] > test_risk.min()
assert test_risk[-1] > test_risk.min() * 1.5
assert 0 < m_opt < mstop_max // 2

test risk: m=0 -> 5.986, min=1.196 at m=60, m=2000 -> 2.039


The training risk decreases at every iteration, as Theorem 1 predicts for
this convex, differentiable loss. The test risk falls from 5.99 at the offset
to a minimum of 1.20 at $m = 60$ and then rises to 2.04 (1.7 times the
minimum) at $m = 2000$, as later updates fit noise. This is one training
sample.

### 1.5 Numerical check: the $\nu$–$m_{\text{stop}}$ trade-off

The same training and test samples are refitted with $\nu = 0.1$ ($m_{\max} =
2000$, test risk every 20 iterations) and $\nu = 0.05$ ($m_{\max} = 4000$,
every 40 iterations), and the test-risk-minimizing iteration of each is
recorded. The predicted scaling is approximately twice as many iterations when
$\nu$ is halved.

In [3]:
def oracle_m_opt(nu, mstop_max, step):
    fit = mb.glmboost(formula, data=data_train, family=mb.Gaussian(),
                       control=mb.boost_control(mstop=mstop_max, nu=nu))
    grid = np.arange(0, mstop_max + 1, step)
    risk = np.array([
        np.mean((np.full(n_test, fit.offset_) - y_test) ** 2) if m == 0
        else np.mean((fit[:m].predict(newdata=data_test, type="response") - y_test) ** 2)
        for m in grid
    ])
    return int(grid[np.argmin(risk)])


m_opt_01 = oracle_m_opt(nu=0.1, mstop_max=2000, step=20)
m_opt_005 = oracle_m_opt(nu=0.05, mstop_max=4000, step=40)
print(f"nu=0.10 -> m_opt={m_opt_01}")
print(f"nu=0.05 -> m_opt={m_opt_005}")
print(f"ratio = {m_opt_005 / m_opt_01:.2f}")

assert 1.5 < m_opt_005 / m_opt_01 < 2.5


nu=0.10 -> m_opt=60
nu=0.05 -> m_opt=120
ratio = 2.00


Halving $\nu$ from 0.1 to 0.05 doubles the test-risk-optimal
$m_{\text{stop}}$, from 60 to 120, in this one sample. The grid spacing (20
and 40 iterations) limits the resolution of the ratio.

## 2. Consistency



### 2.1 The setup

Consistency describes whether $\hat f^{[m_n]}$ (boosting stopped at
$m_n$, which may grow with $n$) converges to the Bayes-optimal $f^*$ as
$n \to \infty$. This is a *statistical* property (about the data-generating process), distinct from the
*algorithmic* convergence of §1 (which is about the optimization).

We work in the following setting:
- $Y = f^*(X) + \varepsilon$, with $\mathbb{E}[\varepsilon \mid X] = 0$ and
  $\operatorname{Var}(\varepsilon \mid X) = \sigma^2$.
- $X = (X_1, \dots, X_p)$ with $p$ potentially growing with $n$.
- Only $S \subset \{1,\dots,p\}$ are "relevant": $f^*(X)$ depends only on
  $\{X_j : j \in S\}$. Let $s = |S|$ be the sparsity.
- The base learners are linear terms `bols(X_j)`.



### 2.2 The L2Boosting consistency theorem

```{admonition} Theorem 3 (L2Boosting consistency)
:class: important

This result is due to {cite}`buhlmann2006boosting`.

Consider the high-dimensional linear model $f^*(x) = \sum_j \beta_j x_j$ with
the number of predictors growing as fast as $\log p_n = O(n^{1-\xi})$ for some
$\xi \in (0,1)$, coefficients whose absolute values have a bounded sum
($\sup_n \sum_j |\beta_j| < \infty$, a form of sparsity), and bounded,
suitably regular predictors and errors. L2Boosting with component-wise linear
base learners, stopped at an iteration $m_n \to \infty$ that grows
sufficiently slowly with $n$, is consistent:

$$
\mathbb{E}_X\big[(\hat f^{[m_n]}(X) - f^*(X))^2\big] \to 0
\quad \text{in probability as } n \to \infty.
$$

The theorem gives no rate, and its stopping iteration is a deterministic
sequence rather than a cross-validated choice.
```

For comparison, the rate $s \log p / n$ is the benchmark for sparse linear
estimation with $s$ nonzero coefficients among $p$: an estimator told the
active set attains $s/n$ (§3.2), and the factor $\log p$ is the price of
searching for the active set. The lasso attains this rate under
restricted-eigenvalue conditions on the design (§3.3); the factor cannot be
removed in the minimax sense without further structural assumptions.

```{figure} ../figures/fig-consistency-rate.svg
:label: fig-consistency-rate
:alt: Estimation error versus sample size on log-log scale. Error decreases at rate s·log(p)/n and approaches zero as n grows (consistency).

Schematic of estimation error against sample size, on log-log axes, for an estimator that attains the benchmark rate $s\log(p)/n$. Consistency means only that the error goes to zero; the theorem above does not establish this rate for L2Boosting.
```



### 2.3 Sparsity and consistency

In the sparse linear setting, **greedy selection** tends to select relevant
variables with large coefficients early. Noise variables tend to enter later.
Cross-validation can stop the path before noise fitting dominates.

```{figure} ../figures/mermaid-f2229555ec5d.svg
:name: mermaid-f2229555ec5d
:alt: Two phases of a boosting path: early iterations select strong signal variables, whereas later iterations increasingly select noise variables.

Early stopping truncates a stylized boosting path after the principal signal has been learned but before later noise-fitting updates dominate.
```

Early stopping is effective when the path separates updates that capture
signal from later updates that fit noise. A larger signal-to-noise ratio can
make this separation clearer.



### 2.4 Consistency with smooth terms

{cite}`buhlmann2003boosting` analyse L2Boosting with a single predictor and a
smoothing-spline base learner of order $r$, applied repeatedly. If $f^*$ lies
in a Sobolev space of smoothness $k \ge r$ (roughly, $k$ square-integrable
derivatives) and the stopping iteration grows at the rate
$m_n = O(n^{2r/(2k+1)})$, the estimator attains

$$
\mathbb{E}\,\frac1n\|\hat f^{[m_n]} - f^*\|^2 = O(n^{-2k/(2k+1)}),
$$

which is the minimax-optimal rate for smoothness $k$, the rate also attained by
smoothing splines and kernel regression tuned to $k$. Boosting a learner of
fixed order $r$ therefore adapts to any higher smoothness $k$ through the
stopping iteration. The result covers one smooth term with a fixed, not
data-chosen, stopping sequence; it does not cover additive models with several
P-spline (`bbs`) terms and component-wise selection.



## 3. Oracle inequalities



### 3.1 Oracle inequalities

An **oracle inequality** bounds the performance of an estimator relative to an
*oracle* that knows the true model structure. For boosting, the oracle knows
which variables are in the active set $S$ and fits OLS on those variables only.
The oracle's error is $O(s/n)$ — the best possible for a linear model with $s$
active variables.

An oracle inequality says: boosting's error is at most a constant factor times
the oracle's error, plus a small price for not knowing $S$.



### 3.2 Components of an oracle comparison

An oracle comparison must include the error from approximating $f^*$ by the
candidate model. For a subset $S'$, let $V_{S'}$ be the span of its design
columns, and define the empirical approximation error

$$
A(S')=\inf_{g\in V_{S'}}\frac1n\|g-f^*\|^2.
$$

If the intercept has already been removed, $V_\varnothing=\{0\}$ and
$A(\varnothing)=\|f^*\|^2/n$. Thus an empty candidate has zero dimension but
usually positive approximation error. A dimension-only infimum cannot bound
prediction error for an arbitrary nonzero $f^*$.

For a fixed design, suppose an oracle projects $y=f^*+\varepsilon$ onto
$V_{S'}$, with $\mathbb E\varepsilon=0$ and
$\operatorname{Cov}(\varepsilon)=\sigma^2 I$. Writing $P_{S'}$ for the
orthogonal projector, the squared bias and variance give

$$
\frac1n\mathbb E\|P_{S'}y-f^*\|^2
=A(S')+\frac{\sigma^2\operatorname{rank}(X_{S'})}{n}.
$$

An oracle told the true active set has $A(S_{\mathrm{true}})=0$ in a correctly
specified linear model. If its $s$ columns are linearly independent, its
expected error is $\sigma^2s/n$. An estimator searching over subsets must
also account for that search. A valid boosting oracle inequality requires a
source theorem specifying the approximation term, complexity penalty, design
conditions, and stopping rule. The assumptions stated in this chapter do not
establish such an inequality, so the oracle benchmark above is not presented
as a finite-sample guarantee for a fitted boosting path.

```{figure} ../figures/fig-oracle-hierarchy.svg
:label: fig-oracle-hierarchy
:alt: Schematic comparison of an oracle rate s/n, a sparse-search rate s log(p)/n, and the full-rank OLS rate p/n.

Schematic estimation rates: an oracle told which $s$ variables matter, a sparse-search benchmark with a $\log p$ factor, and OLS using all $p$ columns. Each comparison requires its own model and design assumptions; the figure is not an oracle inequality for every boosting fit.
```


### 3.3 Comparison with the lasso

The lasso attains the $s \log p / n$ prediction rate under a
restricted-eigenvalue condition on the design: the Gram matrix $X^\top X/n$
is bounded away from singular in the directions of sparse coefficient
vectors, so no small set of columns is nearly collinear. It recovers the
exact active set under the irrepresentable condition, which requires the
irrelevant predictors to be only weakly correlated with the relevant ones
(the coefficients of their regression on the relevant set have $\ell_1$ norm
below 1), together with a lower bound on the smallest nonzero coefficient.

| Property | Lasso | Component-wise L2Boosting |
|---|---|---|
| Prediction result | rate $s \log p / n$ (restricted eigenvalue) | consistency without a rate (Theorem 3) |
| Path | convex problem indexed by $\lambda$ | greedy, indexed by $m_{\text{stop}}$ |
| Exact active-set recovery | under the irrepresentable condition | no result on this page |
| Smooth terms | needs a basis expansion and group penalty | native (`bbs`) |
| Non-Gaussian loss | needs a specialised solver | any differentiable loss with the same algorithm |

Boosting's practical advantage is generality: the same algorithm fits smooth
terms, non-Gaussian losses and distributional regression. The theory for
those extensions is weaker than for the Gaussian linear case (§7.2).



## 4. The geometry of boosting: shrinkage and the lasso connection



### 4.1 Boosting as implicit $\ell_1$ regularization

For centered, mutually orthogonal linear design columns with equal norms,
boosting and $\ell_1$-penalized regression have related coefficient paths.
Consider unpenalized component-wise L2Boosting starting from zero coefficients,
with step length $0<\nu\le1$. Let $k_j(m)$ count the selections of column $j$
through iteration $m$.

Orthogonality implies that the least-squares coefficient obtained by fitting
column $j$ to the current residual is
$\hat\beta_j^{\mathrm{OLS}}-\hat\beta_j$. When $j$ is selected, its coefficient
therefore changes according to

$$
\hat\beta_j^{\mathrm{new}}
=\hat\beta_j+\nu(\hat\beta_j^{\mathrm{OLS}}-\hat\beta_j),
\qquad
\hat\beta_j^{\mathrm{OLS}}-\hat\beta_j^{\mathrm{new}}
=(1-\nu)(\hat\beta_j^{\mathrm{OLS}}-\hat\beta_j).
$$

Starting at zero and applying this recurrence $k_j(m)$ times gives

$$
\hat\beta_j^{[m]}=
\left[1-(1-\nu)^{k_j(m)}\right]\hat\beta_j^{\mathrm{OLS}}.
$$

The coefficient grows geometrically toward its OLS value; the approximation
$\nu k_j(m)\hat\beta_j^{\mathrm{OLS}}$ applies only while
$\nu k_j(m)$ is small.

Greedy selection updates the largest remaining residual correlation. With
these standardized orthogonal columns, the infinitesimal-step limit traces
the lasso coefficient path, up to its parameterization {cite}`efron2004leastangle`.
At finite $\nu$, the boosting path has discrete updates and need not coincide
with the lasso at every stopping iteration.

```{figure} ../figures/fig-coef-paths.svg
:label: fig-coef-paths
:alt: Lasso coefficient paths as lambda decreases and L2Boosting coefficient paths as the stopping iteration increases.

Lasso and L2Boosting paths on the same data. Under a standardized orthogonal design, the paths coincide in the infinitesimal-step limit; finite boosting updates produce a discrete approximation.
```


### 4.2 Forward stagewise, boosting, and the infinitesimal lasso

Forward stagewise regression moves the coefficient of the predictor most
correlated with the current residual by a small fixed amount at each step; it
is component-wise L2Boosting with linear learners and a small step length.
Least angle regression (LARS) computes a piecewise-linear path exactly. For an
orthogonal design, {cite}`efron2004leastangle` show that three paths coincide:
the lasso path, LARS, and forward stagewise regression in the limit of
infinitesimal steps (equivalently, L2Boosting as $\nu \to 0$).

```{figure} ../figures/mermaid-59cc9af5f0a8.svg
:name: mermaid-59cc9af5f0a8
:alt: Forward stagewise regression, infinitesimal-step L2Boosting, and least angle regression converge to the same lasso path under orthogonality.

For orthogonal designs and infinitesimal step sizes, forward stagewise regression, L2Boosting, and LARS trace the lasso coefficient path.
```

At a finite step length, L2Boosting follows this path in discrete steps
(§4.1). For correlated designs the three paths can differ. Boosting extends
the stagewise construction to other losses and nonlinear base learners.



## 5. Degrees of freedom and the AIC justification



### 5.1 Effective degrees of freedom of the boosted predictor

For Gaussian L2Boosting, treating the selected learner sequence as fixed gives
a linear map $\hat y=B_m y$. The degrees of freedom that `aic()` uses are the
trace

$$
\mathrm{df}_{\mathrm{fit}}(m)=\operatorname{tr}(B_m),
$$

as in R mboost; for Binomial and Poisson models each step of $B_m$ is weighted
by the working (IRLS) weights at the current fit. The criteria built on this
trace are in [](model_selection.ipynb). The calculation treats the selected
path as fixed and does not include the extra complexity of choosing that path
from the data.

To distinguish this trace from the training-noise reduction, consider a fixed
linear smoother $\hat y=Sy$, with $y=f^*+\varepsilon$,
$\mathbb E\varepsilon=0$, and
$\operatorname{Cov}(\varepsilon)=\sigma^2 I$. Expanding the residual gives

$$
y-Sy=(I-S)f^*+(I-S)\varepsilon.
$$

The cross-term has expectation zero, hence

$$
\begin{aligned}
\mathbb E\|y-Sy\|^2
&=\|(I-S)f^*\|^2+
  \sigma^2\operatorname{tr}[(I-S)(I-S)^\top]\\
&=\|(I-S)f^*\|^2+
  \sigma^2\left[n-2\operatorname{tr}(S)+\operatorname{tr}(SS^\top)\right].
\end{aligned}
$$

Thus $\operatorname{tr}(2S-SS^\top)$ measures the reduction in the noise
contribution to training residual sum of squares, relative to $n\sigma^2$;
$n - \operatorname{tr}(2S-SS^\top)$ is the residual degrees of freedom of the
smoother. It is a different quantity from $\operatorname{tr}(S)$. mboostrs,
like R mboost, uses its single-learner form $\operatorname{tr}(2H-HH^\top)$
to convert a learner's `df` argument into a penalty $\lambda$
([](penalties_df.ipynb)), and does not use it for AIC. For a fresh response
$y^{\mathrm{new}}=f^*+\varepsilon^{\mathrm{new}}$ at the same design points,
with independent noise of the same covariance,

$$
\mathbb E\|y^{\mathrm{new}}-Sy\|^2
=\|(I-S)f^*\|^2+n\sigma^2+
 \sigma^2\operatorname{tr}(SS^\top).
$$

Subtracting the expected training error gives the squared-error optimism
$2\sigma^2\operatorname{tr}(S)$. For an OLS orthogonal projector $H$ of rank
$p$, $HH^\top=H$; the training noise contribution is $\sigma^2(n-p)$ and
the optimism is $2\sigma^2p$.

When $S$ is symmetric with eigenvalues $\lambda_j\in[0,1]$,

$$
\operatorname{tr}(2S-SS^\top)
=\sum_j(2\lambda_j-\lambda_j^2)\ge\sum_j\lambda_j
=\operatorname{tr}(S).
$$

Equality holds for a projection. The exercise below derives this comparison;
it does not identify the larger trace as the AIC degrees of freedom.

```{list-table} Two smoother traces
:header-rows: 1

* - Smoother $S$
  - Fit degrees of freedom
  - Training-noise reduction / $\sigma^2$
  - Agreement
* - OLS projector $H$
  - $\operatorname{rank}(X)$
  - $\operatorname{rank}(X)$
  - $H^2=H=H^\top$
* - Ridge $H_\lambda$
  - $\operatorname{tr}(H_\lambda)$
  - $\operatorname{tr}(2H_\lambda-H_\lambda^2)$
  - Generally different
* - Fixed boosting path $B_m$
  - $\operatorname{tr}(B_m)$
  - $\operatorname{tr}(2B_m-B_mB_m^\top)$
  - Generally different
```

These expectation identities require a deterministic $S$. Substituting a
hat matrix formed along a data-selected boosting path provides a conditional
complexity approximation, rather than an unbiased correction for the full
adaptive fitting procedure.


### 5.2 AIC as an asymptotic approximation to CV

The AIC criterion

$$
\mathrm{AIC}(m) = n\log(\hat\sigma_m^2) + 2\,\mathrm{df}(m)
$$

estimates the expected log-likelihood on new data, and for parametric models
it is asymptotically equivalent to leave-one-out cross-validation: both
estimate the same out-of-sample loss and select the same model as
$n \to \infty$ under regularity conditions.
In finite samples, AIC tends to be faster but slightly more liberal
(overfit) than CV, especially for small $n$ or complex models. `mboostrs`
therefore recommends `cvrisk` by default and AIC as a fast approximation
(see [](model_selection.ipynb)).



## 6. Theoretical properties of GAMLSS boosting



### 6.1 Consistency of distributional boosting

For GAMLSS boosting (boosting of location, scale and shape models, in which each parameter of the response distribution has its own additive predictor; [](gamlss.ipynb)), consistency requires that *all*
distribution parameters, $(f_1^*, \dots, f_K^*)$, are estimated
consistently. {cite}`mayr2012generalized` introduced cyclic GAMLSS boosting
and studied it by simulation; no consistency theorem for GAMLSS boosting is
stated or cited on this page, so consistency for these models is an open
question here (§7.2).

**Identifiability** requires that the parameters be jointly
identifiable (no two parameter combinations give the same density). For most
families this holds, but for some it is fragile: the Student-t distribution
approaches the Gaussian as its degrees-of-freedom parameter grows, so that
parameter is poorly identified when the data are close to Gaussian. (This
parameter is unrelated to the step length $\nu$.)



### 6.2 Non-cyclic fitting and monotone risk

**Non-cyclic** GAMLSS boosting {cite}`thomas2018gradient` computes a
gradient update for every distribution parameter at each iteration and
applies only the one that most reduces the joint risk. Each candidate is a
gradient step for one parameter, so the argument of Theorem 1 gives a
decrease of the joint risk at every iteration for a sufficiently small step
length. Cyclic boosting updates every parameter in turn, and a later
parameter's update within a cycle can increase the joint risk.

```{figure} ../figures/fig-joint-risk.svg
:label: fig-joint-risk
:alt: Joint risk trajectory for cyclic versus non-cyclic GAMLSS boosting. Non-cyclic decreases at every iteration for a small step length; cyclic may rise slightly within a cycle.

Joint risk under both GAMLSS schedules. Non-cyclic decreases at every iteration for a sufficiently small step length, since it applies only the update that most reduces the joint risk; cyclic can rise within a cycle when a later parameter's update undoes part of an earlier one.
```

Non-cyclic fitting uses a single $m_{\text{stop}}$ to regularize all parameters
and reduces the tuning grid from $K$ dimensions to one.



## 7. Scope of the guarantees



### 7.1 Guaranteed properties

- **Training risk decrease** (Theorem 1): for any convex differentiable loss,
  boosting with small $\nu$ does not increase the empirical risk.
- **Consistency** (Theorem 3): for high-dimensional linear models with
  $\log p_n = O(n^{1-\xi})$ and coefficients of bounded $\ell_1$ norm,
  component-wise L2Boosting stopped at a slowly growing deterministic
  $m_n$ is consistent. No rate is given.
- **Nonparametric rate** (§2.4): for one predictor, a smoothing-spline
  learner of order $r$ and $f^*$ of Sobolev smoothness $k \ge r$,
  L2Boosting stopped at $m_n = O(n^{2r/(2k+1)})$ attains the minimax-optimal
  rate $n^{-2k/(2k+1)}$.
- **Joint risk decrease** (§6.2): non-cyclic GAMLSS updates decrease the
  joint risk at every iteration for a sufficiently small step length.



### 7.2 Properties without a guarantee

- **Variable selection consistency** (recovering the exact true set $S$):
  recovery requires further conditions, such as weak correlation between
  relevant and irrelevant predictors and a lower bound on the smallest nonzero
  coefficient, and no such result for boosting is stated on this page. In practice, some irrelevant variables may be
  selected. Use **stability selection** ([](stability_selection.ipynb)) for
  selection with error control.
- **Oracle inequality** (§3.2): a fixed-model oracle has approximation error
  plus $\sigma^2\operatorname{rank}(X_S)/n$ estimation error; no oracle
  inequality for boosting is established on this page.
- **Inference validity** (p-values, confidence intervals): the standard errors
  from `coef()` and `confint()` are *conditional on the selected model* and
  do **not** account for selection uncertainty. They undercover. Selective
  inference for boosting is an open research problem, and mboostrs does not
  implement it.
- **Global optimality**: boosting is a greedy procedure. It is not guaranteed
  to find the global optimum of the empirical risk (unlike convex methods like
  the lasso). Greedy selection yields interpretability and selection, but the
  exact coefficients depend on selection order.
- **Results for non-Gaussian losses**: the results of §2 are for L2Boosting.
  For Binomial, Poisson, survival and quantile boosting, consistency results
  require further conditions that are harder to verify; see
  {cite}`buhlmann2007boosting` for the general treatment. The quantile (check)
  loss is not differentiable, so Theorem 1 does not apply to it, and its
  training risk path should be checked.

```{list-table} Theoretical guarantees by loss type
:header-rows: 1

* - Loss type
  - Training risk ↓
  - Consistency
  - Oracle inequality
  - Selection
* - Gaussian (L2)
  - ✓ ($0 < \nu \le 1$, §1.2)
  - ✓ (Theorem 3, §2.4)
  - not established
  - approximate
* - Binomial
  - ✓ (small $\nu$)
  - under further conditions
  - not established
  - approximate
* - Poisson
  - ✓ (small $\nu$)
  - under further conditions
  - not established
  - approximate
* - Quantile
  - not guaranteed (non-differentiable loss)
  - under further conditions
  - not established
  - approximate
* - Cox proportional hazards
  - ✓ (small $\nu$)
  - under further conditions
  - not established
  - approximate
* - GAMLSS (cyclic)
  - not guaranteed within a cycle
  - open
  - not established
  - approximate
* - GAMLSS (non-cyclic)
  - ✓ (small $\nu$, §6.2)
  - open
  - not established
  - approximate
```

"approximate" selection: some irrelevant variables may enter; use `stabsel`
for finite-sample error control.



## 8. Practical implications of the theory

1. **Fix $\nu$ small (0.1) and tune $m_{\text{stop}}$.** The $\nu$-$m_{\text{stop}}$
   duality means this is near-optimal; tuning both is wasteful.

2. **Use cross-validation to choose $m_{\text{stop}}$.** AIC is a fast
   approximation but can overfit in small samples. A cross-validated risk
   curve with an interior minimum followed by a rise is the usual pattern and
   gives the clearest choice; flat or monotone curves also occur
   ([](model_selection.ipynb)).

3. **Keep base learners weak (df=4).** This is a practical recommendation,
   not a theorem on this page. A learner with many degrees of freedom reduces
   the residual more per step than a simpler one, so it is selected
   preferentially and much of the fit is made in a few large steps; equal,
   small `df` for every learner makes the selection compare learners on equal
   flexibility.

4. **Inspect selection stability.** Early stopping can leave many learners
   unselected, but the selected set may still contain noise variables.
   Stability selection adds false-positive control under its bound's
   assumptions.

5. **Treat inference as approximate.** Confidence intervals from `confint()`
   are conditional on the selected model and undercover. Report them as
   descriptive, not confirmatory.

6. **For GAMLSS, prefer non-cyclic unless you need per-parameter control.**
   Non-cyclic is simpler (one $m_{\text{stop}}$), decreases the joint risk at
   every iteration for a small step length, and allocates iterations
   adaptively across parameters.

Related pages: [](model_selection.ipynb) ($m_{\text{stop}}$ selection),
[](stability_selection.ipynb) (variable selection with error control),
[](gamlss_framework.ipynb) and [](gamlss.ipynb) (distributional regression).



## Exercises

````{admonition} Exercise: Prove monotone risk decrease (Theorem 1) in detail
:class: dropdown

Let $\rho(y, f)$ be convex and differentiable in $f$, and define the empirical
risk $\mathcal{R}_n(f) = \frac1n \sum_i \rho(y_i, f(x_i))$.  At iteration $m$,
let $u^{[m]} = -\nabla \mathcal{R}_n(\hat f^{[m-1]} )$ be the negative
gradient, and let $\hat g_{j^\star}$ be the base learner that best fits
$u^{[m]}$ in the $L_2$ sense (i.e. the projection of $u^{[m]}$ onto the
learner's column space).

**a)** Write the first-order Taylor expansion of $\mathcal{R}_n$ at
$\hat f^{[m-1]}$ in the direction $g$:

$$
\mathcal{R}_n(\hat f^{[m-1]} + \nu g)
= \mathcal{R}_n(\hat f^{[m-1]}) - \nu\,\langle u^{[m]}, g\rangle
  + o(\nu).
$$

If the Hessian exists and is locally bounded, the remainder is $O(\nu^2)$.

**b)** Show that $\langle u^{[m]}, \hat g_{j^\star} \rangle \ge 0$, and if the
learner space is rich enough to capture some of the gradient (i.e.
$\hat g_{j^\star} \ne 0$), then
$\langle u^{[m]}, \hat g_{j^\star} \rangle > 0$.

**c)** Combine (a) and (b) with $\nu$ sufficiently small to conclude

$$
\mathcal{R}_n(\hat f^{[m]}) \le \mathcal{R}_n(\hat f^{[m-1]}).
$$

Why must $\nu$ be "sufficiently small"?  What goes wrong if $\nu$ is too large?

**d)** Show that a nonzero projected gradient gives strict decrease for a
sufficiently small positive step, without requiring strict convexity.

````

````{admonition} Solution
:class: dropdown

**a)** Differentiability gives a remainder $o(\nu)$ for a fixed direction
$g$. Under the additional assumption of a continuous Hessian, Taylor's
theorem gives:

$$
\mathcal{R}_n(f + \nu g) = \mathcal{R}_n(f)
   + \nu \,\nabla \mathcal{R}_n(f)^\top g
   + \frac{\nu^2}{2}\, g^\top \nabla^2 \mathcal{R}_n(f + \xi \nu g)\, g,
$$

for some $\xi \in [0, 1]$.  Since $\nabla \mathcal{R}_n = -u$,
the linear term is $-\nu \langle u, g \rangle$.  The quadratic term is
$O(\nu^2 \|g\|^2)$ when $\nabla^2 \mathcal{R}_n$ is bounded.

**b)** Because $\hat g_{j^\star}$ is the $L_2$ projection of $u^{[m]}$ onto
the learner's column space, the inner product $\langle u^{[m]}, \hat g_{j^\star} \rangle
= \|\hat g_{j^\star}\|^2 \ge 0$ (the projection and the target have a non-negative
dot product when the projection is obtained by least squares, because the
residual is orthogonal to the projection).  If $\hat g_{j^\star} \ne 0$, then
$\langle u^{[m]}, \hat g_{j^\star} \rangle = \|\hat g_{j^\star}\|^2 > 0$.
This identity uses $H^2 = H$, which holds for an unpenalized learner. A
penalized learner's hat matrix is symmetric and positive semidefinite but not
idempotent, and only the general form below holds.

In general, $\hat g_{j^\star} = H_{j^\star} u^{[m]}$ where $H_{j^\star}$
is the hat matrix of learner $j^\star$, symmetric positive semidefinite.  Then

$$
\langle u^{[m]}, \hat g_{j^\star} \rangle = (u^{[m]})^\top H_{j^\star} u^{[m]}
= \|H_{j^\star}^{1/2} u^{[m]}\|^2 \ge 0.
$$

**c)** From (a) and (b),

$$
\mathcal{R}_n(\hat f^{[m]}) - \mathcal{R}_n(\hat f^{[m-1]})
= -\nu \|H^{1/2} u^{[m]}\|^2 + o(\nu).
$$

For a nonzero projected gradient and sufficiently small positive $\nu$,
the negative linear term dominates the remainder, so the difference is
strictly negative. If the projection is zero, the update is zero.

For a twice-differentiable convex risk, the Hessian is positive
semidefinite, so the second-order term is nonnegative. With a larger step,
this curvature term can outweigh the linear decrease. The local argument
therefore requires a step small enough for the current risk and direction;
it does not show that every fixed choice of $\nu$ decreases every convex loss.

**d)** If $\hat g_{j^\star}\ne0$, the directional derivative equals
$-\langle u,\hat g_{j^\star}\rangle=-\|\hat g_{j^\star}\|^2<0$.
Differentiability then gives strict risk decrease for a sufficiently small
positive step. Strict convexity is unnecessary for this local conclusion;
it also does not, by itself, imply an everywhere positive-definite Hessian.
````

````{admonition} Exercise: The $2H - HH^\top$ df identity for an orthogonal projector
:class: dropdown

Let $S$ be a linear smoother (an $n \times n$ matrix) with
$\hat y = S y$.

**a)** Suppose $S$ is an orthogonal projection matrix:
$S^2 = S = S^\top$.  Show that

$$
\operatorname{tr}(2S - S S^\top) = \operatorname{tr}(S).
$$

**b)** For the OLS hat matrix $H = X(X^\top X)^{-1} X^\top$, verify $H$ is an
orthogonal projection and that $\operatorname{tr}(H) = \operatorname{rank}(X) = p$.

**c)** Let $B_m$ be the L2Boosting hat matrix along a fixed selected sequence:

$$
B_0=0,\qquad B_m=B_{m-1}+\nu H_{j_m^\star}(I-B_{m-1}),
$$

where each $H_j$ is an orthogonal projection. Show that $B_1$ is not a
projection when $0<\nu<1$ and the first selected learner is nonzero. If the
same learner is selected repeatedly, derive $B_m$ and check idempotence.
Give an example with $\nu=1$ and two different learners for which $B_2$ is
a projection.

**d)** Assuming $B_m$ is symmetric with eigenvalues in $[0,1]$, compare
$\operatorname{tr}(2B_m - B_m B_m^\top)$ with $\operatorname{tr}(B_m)$.
Which trace measures the training-noise reduction, and which trace is used
for fit degrees of freedom in `aic()`?

````

````{admonition} Solution
:class: dropdown

**a)** Since $S = S^\top$ and $S^2 = S$, we have $S S^\top = S^2 = S$.
Hence

$$
\operatorname{tr}(2S - S S^\top) = \operatorname{tr}(2S - S)
= \operatorname{tr}(S).
$$

**b)** $H$ is symmetric ($H^\top = H$) and idempotent ($H^2 = H$), so it is
an orthogonal projection onto the column space of $X$.  The trace of a
projection equals its rank, and $\operatorname{rank}(H) = \operatorname{rank}(X) = p$.
Hence $\operatorname{tr}(H) = p$ and
$\operatorname{tr}(2H - HH^\top) = \operatorname{tr}(H) = p$.

**c)** For a nonzero projection $H$, $B_1=\nu H$ and

$$
B_1^2=\nu^2H\ne\nu H=B_1\qquad(0<\nu<1).
$$

Repeated selection of this learner gives
$B_m=[1-(1-\nu)^m]H$. The nonzero eigenvalues are strictly between zero and
one for finite $m$, so this matrix is not idempotent. This example does not
prove a universal condition on all possible selected sequences. For example,
with $\nu=1$ and mutually orthogonal rank-one projectors $H_1,H_2$,

$$
B_2=H_1+H_2(I-H_1)=H_1+H_2,
$$

which is a projection even though the second learner's space does not contain
the first learner's space.

**d)** For a symmetric $B_m$ with eigenvalues $\lambda_j \in [0, 1]$:

$$
\operatorname{tr}(B_m) = \sum \lambda_j, \qquad
\operatorname{tr}(B_m^2) = \sum \lambda_j^2.
$$

Since $\lambda_j^2 \le \lambda_j$ for $\lambda_j \in [0, 1]$,

$$
\operatorname{tr}(2B_m - B_m^2) = \sum (2\lambda_j - \lambda_j^2)
= \sum \lambda_j (2 - \lambda_j) \ge \sum \lambda_j = \operatorname{tr}(B_m).
$$

Equality holds only when every $\lambda_j\in\{0,1\}$, so $B_m$ is an
orthogonal projection. If at least one eigenvalue is strictly between zero
and one, the training-noise reduction trace is strictly larger than the fit
trace. Zero eigenvalues contribute zero to both. `aic()` uses
$\operatorname{tr}(B_m)$; the inequality does not show that this choice
underestimates the AIC degrees of freedom. Adaptive learner selection is a
separate source of complexity omitted by both fixed-matrix calculations.
````

````{admonition} Exercise: Collinearity and the coefficient path
:class: dropdown

Let $x_1,x_2$ be nonzero centered orthogonal vectors, let $x_3=x_1+x_2$,
and take the noiseless response $y=x_1$. Fit unpenalized component-wise
L2Boosting from zero with learners `bols(X1)`, `bols(X2)`, and `bols(X3)`
and step length $\nu=0.1$.

**a)** Show that every coefficient vector satisfying
$\beta_1+\beta_3=1$ and $\beta_2+\beta_3=0$ fits $y$ exactly.

**b)** Compare the residual sums of squares from projecting a residual
$r=a x_1$, with $a\ne0$, onto each of the three learner spaces.
Which learner is selected?

**c)** Derive the first five coefficient updates and their limit.

**d)** Find the minimum-$\ell_2$-norm exact-fit coefficient vector. Does the
component-wise boosting path converge to that vector? Explain what changes
if the response includes observation noise.
````

````{admonition} Solution
:class: dropdown

**a)** Since $x_3=x_1+x_2$,

$$
X\beta=(\beta_1+\beta_3)x_1+(\beta_2+\beta_3)x_2.
$$

Orthogonality makes $x_1,x_2$ linearly independent, so this equals $y=x_1$
exactly when the two displayed constraints hold. Parameterizing by
$t=\beta_3$ gives the family $(\beta_1,\beta_2,\beta_3)=(1-t,-t,t)$.
The fitted vector is unique although these coefficient vectors are not.

**b)** Projection onto $x_1$ captures $r$ exactly and has residual sum of
squares zero. Projection onto $x_2$ is zero and leaves $a^2\|x_1\|^2$.
For $x_3$, the projection coefficient is

$$
\hat\gamma_3=a\frac{\|x_1\|^2}{\|x_1\|^2+\|x_2\|^2},
$$

and the remaining squared error is

$$
a^2\|x_1\|^2-
\frac{a^2\|x_1\|^4}{\|x_1\|^2+\|x_2\|^2}
=\frac{a^2\|x_1\|^2\|x_2\|^2}{\|x_1\|^2+\|x_2\|^2}>0.
$$

Thus $x_1$ is uniquely selected whenever $a\ne0$.

**c)** Every update leaves the residual proportional to $x_1$, so the same
comparison applies at every finite iteration. The recurrence is

$$
\hat\beta_1^{[m]}=\hat\beta_1^{[m-1]}+
\nu(1-\hat\beta_1^{[m-1]}),\qquad
\hat\beta_2^{[m]}=\hat\beta_3^{[m]}=0.
$$

Hence $\hat\beta_1^{[m]}=1-0.9^m$, whose first five values are
$0.1$, $0.19$, $0.271$, $0.3439$, and $0.40951$.
The coefficient vector converges to $(1,0,0)$, and the fitted vector converges
to $y$.

**d)** Among the exact-fit vectors from (a),

$$
\|\beta\|^2=(1-t)^2+t^2+t^2=1-2t+3t^2.
$$

Differentiating gives $t=1/3$, so the minimum-norm solution is
$(2/3,-1/3,1/3)$. The boosting limit $(1,0,0)$ is a different exact-fit
solution. Greedy component-wise boosting therefore has no general guarantee
of selecting the Moore–Penrose minimum-norm coefficient vector.

With noisy $y=x_1+\varepsilon$, the fitted OLS vector is the projection of
$y$ onto $\operatorname{span}(x_1,x_2)$, which generally differs from $x_1$.
The constraints in (a) describe the noiseless signal and need to be replaced
by the coordinates of this projection for a noisy sample. Its learner
sequence depends on the observed correlations and may include ties; exact
collinearity alone does not determine a unique sequence or coefficient limit.
````


## See also

- [](../implementation/parity_and_bugs) — how the convergence and consistency
  guarantees interact with the documented differences from R.
- [](../implementation/testing) — how the finite-difference gradient verifier
  and parity tests protect the numerical properties the theory assumes.
- [](../implementation/performance) — empirical runtime and scalability,
  including measured per-iteration costs for the algorithm the theory analyses.

```{bibliography}
:filter: docname in docnames
```